# Part 2 : Snowflake CoWork

## シナリオ

あなたは**旅行グッズブランドのマーケティング担当**です。
インフルエンサーに商品を提供して SNS 投稿してもらう施策を実施しました。

手元にあるデータは以下です。

| データ | 内容 | 種類 |
|---|---|---|
| `products` | 商品マスタ（5商品） | 構造化 |
| `posts` | インフルエンサーの SNS 投稿（53件） | 構造化 |
| `daily_sales` | 日別売上（305件） | 構造化 |
| `image_features` | 投稿画像を AI で構造化した特徴量（53件） | 画像 → 構造化 |
| 商品スペックシート PDF | 商品の仕様・こだわりポイント（5商品分） | 非構造化 |

これらを Snowflake CoWork で分析し、**「どんな写真が売上に貢献するのか」**を自然言語で探っていきます。

## このノートブックでやること

| # | 内容 |
|---|---|
| 1 | データ確認 |
| 2 | Agent 作成 + Semantic View |
| 3 | CoWork で質問してみる |
| 4 | Semantic View を改善する |
| 5 | PDF から Cortex Search を構築する |
| 6 | 構造化 + 非構造化を横断する質問 |
| 7 | Artifacts（チャート/テーブルの保存・共有） |
| 8 | Document generation + コード実行ツール |
| 9 | Automations（定期実行レポート化） |

> **前提:** `setup.sql` を実行済みであること。

In [ ]:
%%sql -r dataframe_2_1
-- コンテキスト設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE AI_HANDSON_ADAPTIVE_WH;
USE DATABASE AI_HANDSON_DB;
USE SCHEMA ANALYTICS;

---
## 1. データ確認

まず、これから CoWork で分析するデータを確認しておきます。

In [ ]:
%%sql -r dataframe_2_2
-- 商品マスタ(5商品)
SELECT * FROM products ORDER BY product_id;

In [ ]:
%%sql -r dataframe_2_3
-- インフルエンサー投稿(53件)
SELECT
    post_id,
    product_id,
    posted_at,
    likes,
    comments,
    image_path
FROM posts
ORDER BY posted_at;

In [ ]:
%%sql -r dataframe_2_4
-- 日別売上(商品別の合計を確認)
SELECT
    product_id,
    MIN(sale_date)      AS from_date,
    MAX(sale_date)      AS to_date,
    SUM(units_sold)     AS total_units,
    SUM(sales_amount)   AS total_sales
FROM daily_sales
GROUP BY product_id
ORDER BY product_id;

In [ ]:
%%sql -r dataframe_2_5
-- 投稿画像を AI で構造化した特徴量(setup.sql の AI_COMPLETE で生成済み)
SELECT
    image_file,
    photo_main_subject,
    has_person,
    person_gender,
    person_size,
    expression,
    location,
    product_usage,
    color_tone
FROM image_features
ORDER BY image_file;

---
## 2. Agent 作成 + Semantic View

### Semantic View とは

**Semantic View（セマンティックビュー）** は、CoWork が「このデータが何を意味するのか」を理解するためのメタデータ定義です。

- テーブルの意味や **テーブル同士の関係（RELATIONSHIPS）** を定義する
- ビジネス指標（**METRICS**）をあらかじめ定義しておく
- CoWork はこの定義をもとに自然言語の質問を SQL に変換する

まずは Agent を作成し、その中でシンプルな Semantic View を作って接続します。
主要な3テーブル（`products`、`posts`、`daily_sales`）だけで始めて、
どこまで答えられるかを試していきましょう。

### Cortex Agent を作成して Semantic View を接続する

> **注意:** エージェントの作成には権限が必要です。ロールが **ACCOUNTADMIN** になっていることを確認してください。

#### Step 1: Agent を作成する

1. Snowsight 左メニューから **「AI と ML」→「エージェント」** をクリック
2. 右上の **「エージェントを作成」** をクリック
3. 以下を入力

   | 項目 | 入力値 |
   |---|---|
   | エージェントオブジェクト名 | `AI_HANDSON_DB.ANALYTICS.MARKETING_AGENT` |
   | 表示名 | `マーケティングエージェント` |

4. **「エージェントを作成」** をクリック

#### Step 2: 説明・指示を設定する

作成すると Agent の **Overview** が開きます。上部の **「Configuration」** タブをクリックします。

**「一般」サブタブ** で以下を入力

| 項目 | 入力値 |
|---|---|
| 名前 | `インフルエンサー施策の効果分析。投稿データと売上データから、どんな写真が売上に貢献するかを分析します。` |

**「手順」サブタブ** に切り替えて以下を入力

| 項目 | 入力値 |
|---|---|
| 応答の指示 | `可能な限りチャートで回答してください。日本語で回答してください。` |

右上の **「保存」** をクリック

#### Step 3: Semantic View を Autopilot で作成する

1. **「ツール」サブタブ** に切り替える
2. **「構造化データのクエリ」** セクションのドロップダウンから **「新しいセマンティックビューを追加」** を選択
3. セマンティックビューの管理画面が開くので、右上の **「自動パイロットで作成」** をクリック

Autopilot ウィザードが開きます。左サイドバーに4つのステップが表示されます。

---

**(1) コンテキストを入力してください**

今回はコンテキストなしで作成します。右下の **「スキップ」** をクリック

> SQL クエリや Tableau / Power BI ファイルを入力すると、既存の定義を取り込んで Semantic View を作れます。
> 今回はテーブルのメタデータだけで自動生成し、後から改善していきます。

---

**(2) セマンティックビューに名前を付ける**

| 項目 | 入力値 |
|---|---|
| 名前 | `SV_STEP1` |
| 作成する場所 | `AI_HANDSON_DB.ANALYTICS` |

**「→」**（次へ）をクリック

---

**(3) テーブルを選択**

左のツリーから `AI_HANDSON_DB` → `ANALYTICS` を展開し、以下の **3テーブルのみ** にチェックを入れる

| テーブル | 内容 |
|---|---|
| `PRODUCTS` | 商品マスタ（5商品） |
| `POSTS` | インフルエンサーの SNS 投稿（53件） |
| `DAILY_SALES` | 日別売上（305件） |

> **ポイント:** `IMAGE_FEATURES` テーブルはここでは **追加しません**。

**「→」**（次へ）をクリック

---

**(4) 列を選択**

- カラムは **すべて選択** のままでOK
- 画面下部のトグルを確認:
  - **サンプルのメタ化** → オン
  - **AI** → **オン**

**「作成」** をクリック（生成には数分かかります）

---

#### 作成された Semantic View を確認する

生成が完了すると **Semantic Studio** が開きます。
Logical Tables に指定した3テーブルが表示され、各ディメンションに説明文が追加されていることを確認しましょう。

#### Step 4: 作成した Semantic View を Agent に接続する

1. Agent の設定画面に戻る（**「AI と ML」→「エージェント」** から `MARKETING_AGENT` を開く）
2. **「Configuration」** タブ → **「ツール」サブタブ**
3. **「構造化データのクエリ」** セクションの **「セマンティックビューを追加」** をクリック
4. 以下を入力

   | 項目 | 入力値 |
   |---|---|
   | セマンティックビュー | `AI_HANDSON_DB.ANALYTICS.SV_STEP1` |
   | 名前 | `sv_step1` |
   | 説明 | `インフルエンサー投稿と売上の基本データ` |
   | ウェアハウス | `AI_HANDSON_ADAPTIVE_WH` |

5. **「追加」** → 右上の **「保存」** → **「公開」** をクリック

> **重要:** 「保存」だけでは Draft 状態です。**「公開」を押さないと CoWork に反映されません。**
>
> 「公開」ダイアログが表示されたら、すべてデフォルトのまま **「公開」** を押してください。
>
> | 項目 | 説明 | 今回の操作 |
> |---|---|---|
> | **このバージョンを使用する** | チェックを入れると、公開と同時にこのバージョンがアクティブになる | **チェックのまま** |
> | **注意（オプション）** | バージョンのメモ（後から変更履歴を振り返るとき用） | **空欄のまま** |
> | **エイリアス（オプション）** | API からバージョンを名前で参照するための別名 | **空欄のまま** |

---
## 3. CoWork で質問してみる

### CoWork を開く

1. Snowsight 左メニューから **「AI と ML」→「Snowflake CoWork」** をクリック
2. Agent 選択で **`マーケティングエージェント`** を選ぶ

### 質問例（単一テーブルで完結する質問）

```
商品別の売上金額の合計を教えて
```

```
いいね数が多い投稿トップ5を教えて
```


### こんな質問はどうでしょう？

**(1) テーブルを跨ぐ質問**

```
商品カテゴリごとに、SNSのいいね数と売上金額を並べて見せて
```

**(2) 社内 KPI を使う質問**

```
バズスコアが一番高い投稿はどれ?
```

**(3) 画像の中身に関する質問**

```
人物が写っている投稿と写っていない投稿で、売上効果に差はある?
```

---
## 4. Semantic View を改善する

先ほどの Semantic View はそのまま残し、改善版の `SV_STEP2` を新しく作ります。

### 改善するポイント

| # | 追加するもの | 解決する問題 |
|---|---|---|
| 1 | **COMMENT**（説明文）+ **SYNONYMS**（別名） | カラム名だけでは伝わらない意味を補う |
| 2 | **RELATIONSHIPS**（テーブル間の関係） | テーブルを跨いだ質問に答えられるようにする |
| 3 | **FACTS / METRICS**（社内 KPI の計算式） | 社内独自指標(ここでは「バズスコア」)の計算方法を定義 |
| 4 | **AI_VERIFIED_QUERIES**（検証済みクエリ） | よく聞かれる質問に「お墨付きの SQL」を用意する |

さらに **`image_features`（画像特徴量）テーブルを追加**して、画像の中身を絡めた質問にも答えられるようにします。

### FACTS と METRICS の違い

| 区分 | 役割 | 例 |
|---|---|---|
| **DIMENSIONS** | **カテゴリ属性**（グループ化の軸） | `category`, `color_tone`, `has_person` |
| **FACTS** | **行レベルの数値**（1行ごとの値） | `buzz_score = likes × (1 + comments / 50)` |
| **METRICS** | **集計した指標**（SUM / AVG などで集約） | `total_buzz_score = SUM(buzz_score)` |


### AI_VERIFIED_QUERIES（検証済みクエリ）

「この質問にはこの SQL が正解」というペアを登録しておく機能です。
Agent はこれを参考にするため、**同じ質問への回答が安定します**。
`ONBOARDING_QUESTION TRUE` にすると、CoWork の初期画面に質問候補として表示されます。

In [ ]:
%%sql -r dataframe_2_6
-- Semantic View 改善版
CREATE OR REPLACE SEMANTIC VIEW SV_STEP2

  -- TABLES: テーブルごとに主キー・別名・説明文を定義する
  --   image_features を追加して画像の特徴量も扱えるようにする
  TABLES (
    posts
      PRIMARY KEY (post_id)
      WITH SYNONYMS = ('SNS投稿', 'インフルエンサー投稿')
      COMMENT = 'インフルエンサーの SNS 投稿データ。1行が1投稿',
    products
      PRIMARY KEY (product_id)
      WITH SYNONYMS = ('商品マスタ', '商品')
      COMMENT = '旅行グッズブランドの商品マスタ。5商品',
    daily_sales
      PRIMARY KEY (product_id, sale_date)
      WITH SYNONYMS = ('売上', '販売実績')
      COMMENT = '商品ごとの日別売上データ',
    image_features
      PRIMARY KEY (image_file)
      WITH SYNONYMS = ('画像特徴量', '写真の特徴')
      COMMENT = '投稿画像を AI(AI_COMPLETE)で分析して構造化した特徴量。1行が1画像'
  )

  -- RELATIONSHIPS: テーブル間の結合キーを定義する
  RELATIONSHIPS (
    posts_to_products AS
      posts (product_id) REFERENCES products (product_id),
    daily_sales_to_products AS
      daily_sales (product_id) REFERENCES products (product_id),
    posts_to_image_features AS
      posts (image_path) REFERENCES image_features (image_file)
  )

  -- FACTS: 行レベルの計算式を定義する
  --   バズスコアのような社内独自 KPI はここで計算式を明示する
  FACTS (
    posts.buzz_score AS likes * (1 + comments / 50)
      WITH SYNONYMS = ('バズスコア', 'buzz score')
      COMMENT = 'バズスコア: 社内独自のエンゲージメント評価指標。計算式は likes × (1 + comments / 50)。コメントの方がいいねより重みが大きい'
  )

  -- DIMENSIONS: グループ化や絞り込みに使うカテゴリ属性
  --   SYNONYMS で日本語の別名を付けておくと自然言語の質問にマッチしやすくなる
  DIMENSIONS (
    posts.post_id AS post_id
      COMMENT = '投稿ID',
    posts.posted_at AS posted_at
      COMMENT = '投稿日時',
    posts.posted_date AS TO_DATE(posted_at)
      COMMENT = '投稿日(日付のみ)',
    products.product_id AS product_id
      COMMENT = '商品ID',
    products.product_name AS product_name
      WITH SYNONYMS = ('商品名')
      COMMENT = '商品名',
    products.category AS category
      WITH SYNONYMS = ('商品カテゴリ', 'カテゴリ')
      COMMENT = '商品カテゴリ。スーツケース / ポーチ / ネックピロー / 充電器 / パスポートケース のいずれか',
    daily_sales.sale_date AS sale_date
      COMMENT = '売上日',
    image_features.image_file AS image_file
      COMMENT = '画像ファイル名',
    image_features.photo_main_subject AS photo_main_subject
      WITH SYNONYMS = ('写真の主役')
      COMMENT = '写真の主役。商品 / 人物 / 両方 のいずれか',
    image_features.has_person AS has_person
      WITH SYNONYMS = ('人物の有無', '人が写っているか')
      COMMENT = '人物が写っているか(true / false)',
    image_features.person_gender AS person_gender
      COMMENT = '写っている人物の性別。男性 / 女性 / NULL',
    image_features.person_size AS person_size
      WITH SYNONYMS = ('人物の大きさ')
      COMMENT = '写真内での人物の大きさ。大きい / 中 / 小さい / NULL',
    image_features.expression AS expression
      WITH SYNONYMS = ('表情')
      COMMENT = '人物の表情。笑顔 / クール / 自然体 / NULL',
    image_features.location AS location
      WITH SYNONYMS = ('撮影場所', 'ロケーション')
      COMMENT = '撮影場所。自宅 / カフェ / 屋外 / スタジオ / 空港 / ホテル / 電車内 / その他',
    image_features.product_usage AS product_usage
      WITH SYNONYMS = ('商品の使用状態')
      COMMENT = '写真内での商品の使用状態。未使用 / 使用中 / ビフォーアフター',
    image_features.color_tone AS color_tone
      WITH SYNONYMS = ('色調', '色のトーン')
      COMMENT = '画像全体の色調。暖色系 / 寒色系 / パステル'
  )

  -- METRICS: 集計指標を定義する
  --   FACTS で定義した buzz_score を SUM / AVG して集計できる
  METRICS (
    posts.total_likes AS SUM(posts.likes)
      WITH SYNONYMS = ('いいね数')
      COMMENT = 'いいね数の合計',
    posts.total_comments AS SUM(posts.comments)
      WITH SYNONYMS = ('コメント数')
      COMMENT = 'コメント数の合計',
    posts.total_buzz_score AS SUM(posts.buzz_score)
      WITH SYNONYMS = ('バズスコア合計')
      COMMENT = 'バズスコアの合計',
    posts.avg_buzz_score AS AVG(posts.buzz_score)
      WITH SYNONYMS = ('平均バズスコア')
      COMMENT = '1投稿あたりの平均バズスコア。投稿数が異なるグループを比較するときはこちらを使う',
    posts.post_count AS COUNT(posts.post_id)
      COMMENT = '投稿件数',
    daily_sales.total_units_sold AS SUM(daily_sales.units_sold)
      WITH SYNONYMS = ('販売数')
      COMMENT = '販売数の合計',
    daily_sales.total_sales_amount AS SUM(daily_sales.sales_amount)
      WITH SYNONYMS = ('売上', '売上金額')
      COMMENT = '売上金額の合計(円)',
    products.avg_price AS AVG(products.price)
      COMMENT = '平均販売価格(円)'
  )

  COMMENT = 'インフルエンサー投稿画像の特徴量と売上を横断分析するための Semantic View。バズスコア等の社内KPIも定義済み'

  -- AI_VERIFIED_QUERIES: よく聞かれる質問と正解 SQL のペアを登録する
  --   Agent はこれを参考にして同じ質問に安定して答える
  AI_VERIFIED_QUERIES (
    top_buzz_posts AS (
      QUESTION 'バズスコアが一番高い投稿はどれ?'
      VERIFIED_AT 1788393600
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT post_id, product_id, likes, comments, likes * (1 + comments / 50) AS buzz_score
             FROM posts
             ORDER BY buzz_score DESC NULLS LAST
             LIMIT 10'
    ),
    buzz_by_person AS (
      QUESTION '人物が写っている投稿と写っていない投稿で、平均バズスコアに差はある?'
      VERIFIED_AT 1788393600
      ONBOARDING_QUESTION TRUE
      SQL 'SELECT f.has_person,
                  COUNT(p.post_id) AS post_count,
                  AVG(p.likes * (1 + p.comments / 50)) AS avg_buzz_score
             FROM posts AS p
             JOIN image_features AS f ON p.image_path = f.image_file
             GROUP BY f.has_person
             ORDER BY avg_buzz_score DESC NULLS LAST'
    )
  );

### 改善版の Semantic View を確認する

Semantic Studio で `SV_STEP2` を開いて、SV_STEP1 との違いを確認しましょう。

1. Snowsight 左メニュー **「AI と ML」→「Cortex アナリスト」** を開く
2. データベースで `AI_HANDSON_DB` を選択し、`SV_STEP2` をクリック

確認ポイント:
- **Logical Tables** に `IMAGE_FEATURES` が追加されている
- **Relationships** にテーブル間の結合定義がある
- **FACTS** に `buzz_score` の計算式が定義されている
- **Verified Queries** に日本語の質問と SQL ペアが登録されている

### Agent のツールを改善版に差し替える

#### Step 1: 最初の Semantic View を削除する

1. Snowsight 左メニュー **「AI と ML」→「エージェント」** から `MARKETING_AGENT` を開く
2. **「Configuration」** タブ → **「ツール」サブタブ** を開く
3. **「構造化データのクエリ」** セクションで `sv_step1` の横の **×** をクリックして削除

#### Step 2: 改善版を追加する

1. **「セマンティックビューを追加」** をクリック
2. 以下を入力

   | 項目 | 入力値 |
   |---|---|
   | セマンティックビュー | `AI_HANDSON_DB.ANALYTICS.SV_STEP2` |
   | 名前 | `sv_step2` |
   | 説明 | `インフルエンサー投稿画像の特徴量と売上の横断分析。バズスコア等の社内KPIも定義済み` |
   | ウェアハウス | `AI_HANDSON_ADAPTIVE_WH` |

3. **「追加」** → 右上の **「保存」** → **「公開」** をクリック

#### Step 3: 同じ質問をもう一度投げる

CoWork に戻り、**新しいチャット**を開いて先ほど失敗した質問を再度投げます。

```
商品カテゴリごとに、SNSのいいね数と売上金額を並べて見せて
```

```
バズスコアが一番高い投稿はどれ?
```

```
人物が写っている投稿と写っていない投稿で、平均バズスコアに差はある?
```

> **確認ポイント**
> - 先ほどうまくいかなかった質問に、正しく回答できるようになったか
> - バズスコアが `likes × (1 + comments / 50)` で計算されているか（生成 SQL を確認）
> - `posts` と `image_features` を **JOIN した SQL** が自動生成されているか

### さらに深掘りしてみる

```
バズスコアが高い投稿に共通する写真の特徴を教えて
```

```
商品カテゴリ別に、最も効果的な色のトーンはどれ?
```

```
撮影場所別の平均バズスコアを比較して
```

---
## 5. PDF から Cortex Search を構築する（GUI）

次は**非構造化データ（PDF）**を CoWork から検索できるようにします。

> PDF は `data/pdf/` に配置し、`setup.sql` の Step 3 でステージへコピー済みです。

Cortex Search の GUI ウィザードは、**PDF の解析 → チャンク分割 → インデックス作成**まで一気通貫で行えます。
（SQL での作成も可能です）

```
PDF（ステージ上）
  ↓ GUI ウィザードが自動で処理
  ↓  ① テキスト抽出（AI_PARSE_DOCUMENT 相当）
  ↓  ② チャンク分割（SPLIT_TEXT 相当）
  ↓  ③ 検索インデックス作成
Cortex Search Service 完成 → CoWork から PDF の内容を検索できる
```

---

### Step 1: 新しいサービス

Snowsight 左メニュー **「AI と ML」→「Cortex 検索」** を開き、右上の **「+ Cortex 検索サービス」** をクリックします。

**「新規 Cortex 検索サービスを作成しましょう」** 画面が表示されます。

| 項目 | 設定値 |
|---|---|
| **ウェアハウス** | `AI_HANDSON_ADAPTIVE_WH` |
| **ビルド先（データベース.スキーマ）** | `AI_HANDSON_DB.ANALYTICS` |
| **ビルド名** | `PRODUCT_SPEC_SEARCH` |

右下の **「次へ」** をクリック

---

### Step 2: データを追加

**「インデックスを作成するデータを選択」** 画面が表示されます。

| 項目 | 設定値 |
|---|---|
| **ソース種別（タブ）** | **「ステージ（プレビュー）」** を選択 |
| **ステージ** | `AI_HANDSON_DB` → `ANALYTICS` → `DATA_STAGE` |
| **フォルダ** | `data` → `pdf` を展開 |
| **ファイル選択** | PDF ファイル **5 件すべてにチェック** |

右下の **「次へ」** をクリック

---

### Step 3: 列の設定（解析戦略）

**「ドキュメントの解析戦略を選択」** 画面が表示されます。

| 選択肢 | 説明 | 選択 |
|---|---|---|
| マルチモーダル構造化 | 画像・表などの構造も含めて解析 | |
| **テキスト組み込み** | テキストを抽出してチャンク分割 | **← こちらを選択** |

右下の **「次へ」** をクリック

---

### Step 4: 設定配置（自動処理を構成）

**「自動処理を構成」** 画面が表示されます。

| 項目 | 設定値 |
|---|---|
| **対象（データベース.スキーマ）** | `AI_HANDSON_DB.ANALYTICS` |
| **チャンキング方式** | **サイズ別のチャンク** |
| **チャンクサイズ** | `300` |
| **チャンクの重複** | `50` |

右下の **「次へ」** をクリック

---

### Step 5: インデックス作成を確認

**「検索サービスを確認」** 画面が表示されます。

| 項目 | 設定値 |
|---|---|
| **ターゲットラグ** | `1 hour` |

設定内容を確認し、右下の **「作成」** をクリック

> 作成には数分かかります。ステータスが **ACTIVE** になるまで待ちましょう。  
Activeになったら、データプレビューが確認できます。

---
## 6. 構造化 + 非構造化を横断する質問

### Agent に Cortex Search Service を追加する

1. Snowsight 左メニュー **「AI と ML」→「エージェント」** から `MARKETING_AGENT` を開く
2. **「Configuration」** タブ → **「ツール」サブタブ** を開く
3. **「ドキュメントと非構造化データを使用」** セクションの **「検索サービスを追加」** をクリック
4. **「ツールを追加: Cortex 検索」** ダイアログが表示されます。以下を入力

   | 項目 | 入力値 |
   |---|---|
   | **検索サービス** | `AI_HANDSON_DB.ANALYTICS.PRODUCT_SPEC_SEARCH` |
   | **名前** | `product_spec_search` |
   | **説明** | `商品スペックシート。素材・サイズ・洗濯・こだわりポイントなどの仕様情報を検索する` |

   > **「高度な構成」** 以降の項目はデフォルトのままで OK です

5. **「追加」** → 右上の **「保存」** → **「公開」** をクリック

> **重要:** 「保存」だけでは Draft 状態です。**「公開」を押さないと CoWork に反映されません。**

これで Agent は **Semantic View（構造化）** と **Search Service（非構造化）** の2つのツールを持ちました。
Agent が質問に応じて**どちらを使うかを自分で判断**します。

### CoWork で試してみる

Snowsight 左メニュー **「AI と ML」→「CoWork」** を開き、**新しいチャット**を開始します。

質問例

#### 非構造化データ（PDF）だけで答えられる質問

```
ネックピローの素材と洗濯方法を教えて
```

####  両方を横断する質問

```
売上が一番良かった商品の、素材やサイズなどのスペックを教えて
```

```
バズスコアが低い商品について、インフルエンサーにどういう投稿をしてもらったらバズれるか、商品の特徴から分析して
```

---
## 7. Artifacts（チャート/テーブルの保存・共有）


#### Step 1: Artifact として保存する

1. これまでの質問の回答から、気に入ったチャートを1つ選ぶ
2. チャート右上の **「Save」** をクリック

#### Step 2: Artifacts ハブで確認する

1. 左メニューの **「Artifacts」** を開く
2. **「保存済み」** タブに、保存したチャートが表示されることを確認
3. タイルをクリックすると展開表示される

#### Step 3: フォローアップ質問をしてみる

1. 展開した Artifact から追加の質問を投げる
2. → **元の会話とは別の新しいスレッド**が始まり、Artifact の文脈を引き継いだまま深掘りできます

---
## 8. Document generation + コード実行ツール

> **リリース状況: Preview**

### まず: コード実行ツールを有効化する

1. **「AI と ML」→「エージェント」** から `MARKETING_AGENT` を開く
2. **「Configuration」** タブ → **「ツール」サブタブ**
3. **「コード実行ツール」** のトグルを **オン** にする
4. 右上の **「保存」** → **「公開」** をクリック

CoWork は分析結果を**そのまま配れるファイル**に変換できます。
裏側では Agent が Python コードを書いて、隔離されたサンドボックスで実行しています。

### PowerPoint を自動生成する

CoWork のチャットで以下のように依頼します。

```
ここまでの分析結果を、マーケティング会議向けのPowerPointにまとめて。
売上上位商品、バズスコアが高い写真の特徴、次の施策への提言を含めてください。
```

---
## 9. Automations（定期実行レポート化）

> **リリース状況: Public Preview**

**Automation** は、Agent への質問を**定期実行されるレポート**に変える機能です。
実行のたびに最新データで質問が再実行され、結果がメールで届きます。

### やってみる

#### Step 1: Automations タブから作成する

1. CoWork 左メニューの **「Automations」** タブを開く
2. **「Create in chat」** をクリック
3. チャット画面が開くので、入力されたプロンプトを実行し、その後の指示に従う

もしくは直接以下のような指示をすることでも設定が可能

```
先週のSNS投稿と売上の傾向をまとめて、毎週月曜の朝9時に送って
```


#### Step 2: 作成した Automation を確認する

**「Automations」** タブに戻ると、登録したレポートが一覧に表示されます。

- スケジュール・配信先・ステータスを確認
- ここから**編集・一時停止・再開・削除**ができます

---
## クリーンアップ

ハンズオン終了後の環境削除は **cleanup.sql** を実行してください。

---
## まとめ

この Part 2 では、CoWork を「素の Agent」から「実運用レベルの Agent」へ育てる流れを体験しました。

| ステップ | やったこと | 得られたこと |
|---|---|---|
| 2 | シンプルな Semantic View で質問 | 単一テーブルの質問には答えられることを確認 |
| 3 | さまざまなパターンで質問 | 定義が足りないと答えられない/間違うことを体感 |
| 4 | COMMENT・RELATIONSHIPS・FACTS/METRICS・検証済みクエリを追加 | 同じ質問に正しく答えられるようになった |
| 5 | PDF から Cortex Search を構築 | 非構造化データも検索できるようになった |
| 6 | 構造化 + 非構造化を横断する質問 | Agent が複数ツールを使い分けて統合回答するようになった |
| 7 | Artifacts | 分析結果を保存・共有できるようになった |
| 8 | Document generation + コード実行 | PowerPoint やファイル成果物を自動生成できるようになった |
| 9 | Automations | 定期実行レポートとして自動化できるようになった |

### 一番のポイント

**CoWork の回答精度は、Semantic View の作り込みでほぼ決まります。**

テーブルをつないで、社内の言葉と計算式を定義する。
この地道な作業が、そのまま回答精度に跳ね返ってきます。

### 明日から試すなら

1. 自社でよく聞かれる質問を5つ書き出す
2. その質問に必要なテーブルだけで小さく Semantic View を作る
3. 答えられなかった質問について、足りない定義（COMMENT / RELATIONSHIPS / METRICS）を足す
4. 安定して答えられるようになった質問を AI_VERIFIED_QUERIES に登録する